# 02 - Train 5 Transfer Success Models with MLflow


- load the dataset
- remove leakage columns
- validate with cross-validation
- tune hyperparameters with `GridSearchCV`
- log all runs to MLflow
- compare training and test performance


## 1) Imports


In [32]:
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


## 2) Load dataset


In [37]:
ROOT = Path.cwd().resolve().parent
DATA_PATH = ROOT / 'data' / 'processed' / 'transfer_modeling_dataset.csv'

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
df.head()
df.info()
df.describe()

Dataset shape: (4995, 46)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4995 entries, 0 to 4994
Data columns (total 46 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   transfer_row_id               4995 non-null   int64  
 1   player_id                     4995 non-null   int64  
 2   transfer_date                 4995 non-null   object 
 3   transfer_season               4995 non-null   object 
 4   from_club_id                  4995 non-null   int64  
 5   to_club_id                    4995 non-null   int64  
 6   from_club_name                4995 non-null   object 
 7   to_club_name                  4995 non-null   object 
 8   transfer_fee_eur              4995 non-null   float64
 9   market_value_at_transfer_eur  4995 non-null   float64
 10  age_at_transfer               4995 non-null   float64
 11  primary_position              4995 non-null   object 
 12  secondary_position            4995 n

,transfer_row_id,player_id,from_club_id,to_club_id,transfer_fee_eur,market_value_at_transfer_eur,age_at_transfer,height_cm,pre_minutes_total,pre_goals_total,...,has_api_prev_standing,has_api_current_form,post_minutes_avg_per_season,post_goals_per90_avg,post_assists_per90_avg,market_value_end_window_eur,criteria_minutes,criteria_value,criteria_position_kpi,transfer_success
count,4995.000000,4.995000e+03,4995.000000,4995.000000,4.995000e+03,4.995000e+03,4995.000000,4995.000000,4995.000000,4995.000000,...,4995.0,4995.0,4995.000000,4995.000000,4995.000000,4.837000e+03,4995.000000,4995.000000,4995.000000,4995.000000
mean,2497.000000,4.707731e+05,3973.752553,1440.644645,2.558456e+06,9.811261e+05,24.263315,183.332032,792.194995,1.349550,...,0.0,0.0,805.611695,0.119040,0.086151,2.194051e+06,0.417417,0.752352,0.242042,0.482082
std,1442.076628,2.327887e+05,8970.000937,2964.335196,8.744098e+06,3.752232e+05,3.962049,6.753847,1010.977728,3.170994,...,0.0,0.0,681.159139,0.183334,0.164256,1.267602e+06,0.493182,0.431689,0.428362,0.499729
min,0.000000,3.333000e+03,1.000000,3.000000,0.000000e+00,2.500000e+04,15.745379,164.500000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,2.500000e+04,0.000000,0.000000,0.000000,0.000000
25%,1248.500000,2.908620e+05,331.000000,252.000000,0.000000e+00,7.000000e+05,21.284052,178.000000,0.000000,0.000000,...,0.0,0.0,180.000000,0.000000,0.000000,9.000000e+05,0.000000,1.000000,0.000000,0.000000
50%,2497.000000,4.598420e+05,868.000000,621.000000,0.000000e+00,1.225000e+06,23.627653,184.000000,240.000000,0.000000,...,0.0,0.0,719.500000,0.039578,0.040577,2.500000e+06,0.000000,1.000000,0.000000,0.000000
75%,3745.500000,6.325900e+05,3329.000000,1108.000000,2.500000e+05,1.225000e+06,26.818618,188.000000,1442.000000,1.000000,...,0.0,0.0,1269.708300,0.179296,0.130751,3.450000e+06,1.000000,1.000000,0.000000,1.000000
max,4994.000000,1.191154e+06,130650.000000,23826.000000,1.270000e+08,1.225000e+06,40.243668,200.500000,4716.000000,50.000000,...,0.0,0.0,3300.000000,2.142857,3.461539,3.450000e+06,1.000000,1.000000,1.000000,1.000000


## 3) Check missing values

We do not drop rows. Missing values are handled in the preprocessing pipeline.


In [16]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing

to_club_current_season_ppg     4995
to_club_prev_season_rank       4995
from_club_avg_age              1319
from_league_code               1319
from_club_squad_size           1319
market_value_end_window_eur     158
dtype: int64

## 4) Keep only features available at transfer time

This avoids target leakage from future information.


In [17]:
target_col = 'transfer_success'

leakage_cols = [
    'transfer_row_id',
    'player_id',
    'transfer_date',
    'from_club_name',
    'to_club_name',
    'post_minutes_avg_per_season',
    'post_goals_per90_avg',
    'post_assists_per90_avg',
    'market_value_end_window_eur',
    'criteria_minutes',
    'criteria_value',
    'criteria_position_kpi',
]

feature_cols = [col for col in df.columns if col not in leakage_cols + [target_col]]
X = df[feature_cols].copy()
y = df[target_col].copy()

print('Features before dropping all-null columns:', len(feature_cols))

Features before dropping all-null columns: 33


## 4.1) Drop columns that are completely empty

These API placeholder columns are currently empty for all rows, so we remove them from training.


In [18]:
all_null_cols = [col for col in X.columns if X[col].isna().all()]
print('All-null columns:', all_null_cols)

X = X.drop(columns=all_null_cols)
feature_cols = X.columns.tolist()
print('Features after dropping all-null columns:', len(feature_cols))

All-null columns: ['to_club_prev_season_rank', 'to_club_current_season_ppg']
Features after dropping all-null columns: 31


## 5) Train/test split

The holdout test set is kept separate until the end.


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Train target distribution:')
print(y_train.value_counts(normalize=True).sort_index())

Train shape: (3996, 31)
Test shape: (999, 31)
Train target distribution:
transfer_success
0    0.518018
1    0.481982
Name: proportion, dtype: float64


## 6) Preprocessing pipeline

- numeric: median imputation + scaling
- categorical: most frequent imputation + one-hot encoding


In [20]:
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

print('Numeric features:', len(numeric_features))
print('Categorical features:', len(categorical_features))

Numeric features: 24
Categorical features: 7


## 7) Why these 5 models?

We use five models with different behavior:

- `DummyClassifier`: baseline
- `LogisticRegression`: simple linear model
- `RandomForestClassifier`: strong tree ensemble
- `SVC`: nonlinear margin-based model
- `DecisionTreeClassifier`: interpretable tree model


## 8) Define 5 models and tuning parameters

In [21]:
model_configs = {
    'baseline_dummy': {
        'model': DummyClassifier(),
        'param_grid': {
            'model__strategy': ['most_frequent', 'prior']
        },
    },
    'logistic_regression': {
        'model': LogisticRegression(max_iter=2000, random_state=42),
        'param_grid': {
            'model__C': [0.1, 1.0, 10.0],
            'model__solver': ['lbfgs'],
        },
    },
    'random_forest': {
        'model': RandomForestClassifier(random_state=42, n_jobs=1),
        'param_grid': {
            'model__n_estimators': [100, 300],
            'model__max_depth': [None, 8, 12],
            'model__min_samples_split': [2, 10],
        },
    },
    'svm_rbf': {
        'model': SVC(kernel='rbf', probability=True, random_state=42),
        'param_grid': {
            'model__C': [0.5, 1.0, 2.0],
            'model__gamma': ['scale', 'auto'],
        },
    },
    'decision_tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'param_grid': {
            'model__max_depth': [4, 6, 8, None],
            'model__min_samples_split': [2, 10, 20],
        },
    },
}

model_configs

{'baseline_dummy': {'model': DummyClassifier(),
  'param_grid': {'model__strategy': ['most_frequent', 'prior']}},
 'logistic_regression': {'model': LogisticRegression(max_iter=2000, random_state=42),
  'param_grid': {'model__C': [0.1, 1.0, 10.0], 'model__solver': ['lbfgs']}},
 'random_forest': {'model': RandomForestClassifier(n_jobs=1, random_state=42),
  'param_grid': {'model__n_estimators': [100, 300],
   'model__max_depth': [None, 8, 12],
   'model__min_samples_split': [2, 10]}},
 'svm_rbf': {'model': SVC(probability=True, random_state=42),
  'param_grid': {'model__C': [0.5, 1.0, 2.0],
   'model__gamma': ['scale', 'auto']}},
 'decision_tree': {'model': DecisionTreeClassifier(random_state=42),
  'param_grid': {'model__max_depth': [4, 6, 8, None],
   'model__min_samples_split': [2, 10, 20]}}}

## 9) Validation setup

We use 3-fold stratified cross-validation on the training set.


In [22]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv

StratifiedKFold(n_splits=3, random_state=42, shuffle=True)

## 10) MLflow setup

To view the runs:

```bash
mlflow ui
```

Then open `http://127.0.0.1:5000`.


In [23]:
EXPERIMENT_NAME = 'transfer_success_simple_models'
mlflow.set_experiment(EXPERIMENT_NAME)
print('MLflow experiment:', EXPERIMENT_NAME)

MLflow experiment: transfer_success_simple_models


## 11) Helper functions

We calculate both standard metrics and business-style metrics.

Business interpretation used here:
- false positive: club thinks a transfer will succeed, but it does not
- false negative: club misses a transfer that could have succeeded

Business metrics in this notebook:
- `success_precision`: how often predicted successful transfers are really successful
- `missed_success_rate`: how often we miss true successful transfers


In [34]:
def evaluate_predictions(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    standard_metrics = {
        'train_or_test_accuracy': accuracy_score(y_true, y_pred),
        'train_or_test_precision': precision_score(y_true, y_pred, zero_division=0),
        'train_or_test_recall': recall_score(y_true, y_pred, zero_division=0),
        'train_or_test_f1': f1_score(y_true, y_pred, zero_division=0),
        'train_or_test_roc_auc': roc_auc_score(y_true, y_proba),
    }

    business_metrics = {
        'success_precision': tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        'missed_success_rate': fn / (fn + tp) if (fn + tp) > 0 else 0.0,
        'false_positive_rate': fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        'captured_success_rate': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
    }

    counts = {
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

    return standard_metrics, business_metrics, counts


def prefix_metrics(metrics_dict, prefix):
    return {f'{prefix}_{k}': v for k, v in metrics_dict.items()}


def build_pipeline(model):
    return Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', model),
        ]
    )


def train_tune_validate_log(model_name, model, param_grid):
    pipeline = build_pipeline(model)

    baseline_cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='f1',
        n_jobs=1,
    )

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='f1',
        n_jobs=1,
    )
    grid_search.fit(X_train, y_train)

    best_pipeline = grid_search.best_estimator_

    train_pred = best_pipeline.predict(X_train)
    train_proba = best_pipeline.predict_proba(X_train)[:, 1]
    train_standard, train_business, train_counts = evaluate_predictions(y_train, train_pred, train_proba)

    test_pred = best_pipeline.predict(X_test)
    test_proba = best_pipeline.predict_proba(X_test)[:, 1]
    test_standard, test_business, test_counts = evaluate_predictions(y_test, test_pred, test_proba)

    result = {
        'model_name': model_name,
        'cv_f1_mean_before_tuning': baseline_cv_scores.mean(),
        'cv_f1_std_before_tuning': baseline_cv_scores.std(),
        'best_cv_f1_after_tuning': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        **prefix_metrics(train_standard, 'train'),
        **prefix_metrics(train_business, 'train'),
        **prefix_metrics(test_standard, 'test'),
        **prefix_metrics(test_business, 'test'),
        **prefix_metrics(train_counts, 'train'),
        **prefix_metrics(test_counts, 'test'),
    }

    mlflow_metrics = {
        'cv_f1_mean_before_tuning': baseline_cv_scores.mean(),
        'cv_f1_std_before_tuning': baseline_cv_scores.std(),
        'best_cv_f1_after_tuning': grid_search.best_score_,
        **prefix_metrics(train_standard, 'train'),
        **prefix_metrics(train_business, 'train'),
        **prefix_metrics(test_standard, 'test'),
        **prefix_metrics(test_business, 'test'),
    }

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('target_column', target_col)
        mlflow.log_param('n_features', len(feature_cols))
        mlflow.log_param('train_rows', len(X_train))
        mlflow.log_param('test_rows', len(X_test))

        for key, value in model.get_params().items():
            mlflow.log_param(f'model__{key}', value)

        for key, value in grid_search.best_params_.items():
            mlflow.log_param(f'best__{key}', value)

        mlflow.log_metrics(mlflow_metrics)
        mlflow.sklearn.log_model(best_pipeline, name='model')

    return result, best_pipeline


## 12) Train, validate, tune, and log all 5 models


In [35]:
results = []
trained_pipelines = {}

for model_name, config in model_configs.items():
    print(f'Training: {model_name}')
    result, pipeline = train_tune_validate_log(
        model_name=model_name,
        model=config['model'],
        param_grid=config['param_grid'],
    )
    results.append(result)
    trained_pipelines[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values(by='test_train_or_test_f1', ascending=False).reset_index(drop=True)
results_df

Training: baseline_dummy


2026/04/26 20:13:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Training: logistic_regression


2026/04/26 20:14:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Training: random_forest


...

Training: svm_rbf


2026/04/26 20:16:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Training: decision_tree


2026/04/26 20:16:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


,model_name,cv_f1_mean_before_tuning,cv_f1_std_before_tuning,best_cv_f1_after_tuning,best_params,train_train_or_test_accuracy,train_train_or_test_precision,train_train_or_test_recall,train_train_or_test_f1,train_train_or_test_roc_auc,...,test_false_positive_rate,test_captured_success_rate,train_tn,train_fp,train_fn,train_tp,test_tn,test_fp,test_fn,test_tp
0,random_forest,0.682625,0.010345,0.688824,"{'model__max_depth': None, 'model__min_samples...",1.000000,1.000000,1.000000,1.000000,1.000000,...,0.239845,0.655602,2070,0,0,1926,393,124,166,316
1,decision_tree,0.604199,0.023662,0.673774,"{'model__max_depth': 4, 'model__min_samples_sp...",0.681181,0.655386,0.713915,0.683400,0.748580,...,0.357834,0.680498,1347,723,551,1375,332,185,154,328
2,logistic_regression,0.667210,0.006123,0.667210,"{'model__C': 1.0, 'model__solver': 'lbfgs'}",0.716967,0.723692,0.667705,0.694572,0.790014,...,0.268859,0.605809,1579,491,640,1286,378,139,190,292
3,svm_rbf,0.664603,0.003824,0.680048,"{'model__C': 2.0, 'model__gamma': 'scale'}",0.825325,0.859906,0.761682,0.807819,0.909653,...,0.249516,0.589212,1831,239,459,1467,388,129,198,284
4,baseline_dummy,0.000000,0.000000,0.000000,{'model__strategy': 'most_frequent'},0.518018,0.000000,0.000000,0.000000,0.500000,...,0.000000,0.000000,2070,0,1926,0,517,0,482,0


## 13) Model comparison table

This table focuses on the metrics you will most likely discuss in the report.


In [26]:
comparison_cols = [
    'model_name',
    'cv_f1_mean_before_tuning',
    'best_cv_f1_after_tuning',
    'train_train_or_test_f1',
    'test_train_or_test_f1',
    'test_train_or_test_accuracy',
    'test_train_or_test_recall',
    'test_train_or_test_roc_auc',
    'test_success_precision',
    'test_missed_success_rate',
    'best_params',
]

results_df[comparison_cols]

,model_name,cv_f1_mean_before_tuning,best_cv_f1_after_tuning,train_train_or_test_f1,test_train_or_test_f1,test_train_or_test_accuracy,test_train_or_test_recall,test_train_or_test_roc_auc,test_success_precision,test_missed_success_rate,best_params
0,random_forest,0.682625,0.688824,1.000000,0.685466,0.709710,0.655602,0.777073,0.718182,0.344398,"{'model__max_depth': None, 'model__min_samples..."
1,decision_tree,0.604199,0.673774,0.683400,0.659296,0.660661,0.680498,0.705559,0.639376,0.319502,"{'model__max_depth': 4, 'model__min_samples_sp..."
2,logistic_regression,0.667210,0.667210,0.694572,0.639650,0.670671,0.605809,0.732574,0.677494,0.394191,"{'model__C': 1.0, 'model__solver': 'lbfgs'}"
3,svm_rbf,0.664603,0.680048,0.807819,0.634637,0.672673,0.589212,0.734402,0.687651,0.410788,"{'model__C': 2.0, 'model__gamma': 'scale'}"
4,baseline_dummy,0.000000,0.000000,0.000000,0.000000,0.517518,0.000000,0.500000,0.000000,1.000000,{'model__strategy': 'most_frequent'}


## 14) Best model summary


In [27]:
best_row = results_df.iloc[0]
best_model_name = best_row['model_name']
best_pipeline = trained_pipelines[best_model_name]

print('Best model:', best_model_name)
print('Best parameters:', best_row['best_params'])
best_row

Best model: random_forest
Best parameters: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 300}


model_name                                                           random_forest
cv_f1_mean_before_tuning                                                  0.682625
cv_f1_std_before_tuning                                                   0.010345
best_cv_f1_after_tuning                                                   0.688824
best_params                      {'model__max_depth': None, 'model__min_samples...
train_train_or_test_accuracy                                                   1.0
train_train_or_test_precision                                                  1.0
train_train_or_test_recall                                                     1.0
train_train_or_test_f1                                                         1.0
train_train_or_test_roc_auc                                                    1.0
train_success_precision                                                        1.0
train_missed_success_rate                                                      0.0
trai

## 15) Overfitting check

Comparing training and test F1 scores gives a quick view of overfitting.


In [28]:
overfit_view = results_df[['model_name', 'train_train_or_test_f1', 'test_train_or_test_f1']].copy()
overfit_view['f1_gap'] = overfit_view['train_train_or_test_f1'] - overfit_view['test_train_or_test_f1']
overfit_view.sort_values(by='f1_gap', ascending=False)

,model_name,train_train_or_test_f1,test_train_or_test_f1,f1_gap
0,random_forest,1.000000,0.685466,0.314534
3,svm_rbf,0.807819,0.634637,0.173183
2,logistic_regression,0.694572,0.639650,0.054922
1,decision_tree,0.683400,0.659296,0.024103
4,baseline_dummy,0.000000,0.000000,0.000000


## 16) Error analysis for the best model

We inspect false positives and false negatives, because they matter differently for the stakeholder.


In [29]:
test_analysis = X_test.copy()
test_analysis['actual'] = y_test.values
test_analysis['predicted'] = best_pipeline.predict(X_test)
test_analysis['predicted_proba'] = best_pipeline.predict_proba(X_test)[:, 1]

false_positives = test_analysis[(test_analysis['actual'] == 0) & (test_analysis['predicted'] == 1)].copy()
false_negatives = test_analysis[(test_analysis['actual'] == 1) & (test_analysis['predicted'] == 0)].copy()

print('False positives:', len(false_positives))
print('False negatives:', len(false_negatives))

False positives: 124
False negatives: 166


In [30]:
false_positives[['transfer_season', 'primary_position', 'age_at_transfer', 'market_value_at_transfer_eur', 'predicted_proba']].head(10)

,transfer_season,primary_position,age_at_transfer,market_value_at_transfer_eur,predicted_proba
4943,22/23,Midfield,17.612595,500000.0,0.510000
1629,23/24,Midfield,27.548254,1225000.0,0.510000
1912,23/24,Attack,29.823408,1225000.0,0.713333
4689,22/23,Attack,23.060917,300000.0,0.646667
54,23/24,Attack,29.002054,1225000.0,0.573333
3878,22/23,Attack,22.461329,1225000.0,0.770000
4339,22/23,Defender,27.337440,1225000.0,0.706667
4961,22/23,Defender,18.428474,500000.0,0.580000
1662,23/24,Midfield,18.223135,400000.0,0.506667
4002,22/23,Attack,24.495550,1225000.0,0.553333


In [31]:
false_negatives[['transfer_season', 'primary_position', 'age_at_transfer', 'market_value_at_transfer_eur', 'predicted_proba']].head(10)

,transfer_season,primary_position,age_at_transfer,market_value_at_transfer_eur,predicted_proba
4848,22/23,Midfield,18.899384,600000.0,0.250000
3573,22/23,Defender,29.639973,500000.0,0.233333
186,23/24,Defender,24.391512,1225000.0,0.386667
4724,22/23,Midfield,22.398357,1225000.0,0.326667
2342,23/24,Attack,21.546886,1000000.0,0.476667
4610,22/23,Goalkeeper,22.844627,1225000.0,0.466667
4875,22/23,Defender,19.041752,100000.0,0.203333
3417,22/23,Midfield,20.030117,500000.0,0.270000
3484,22/23,Midfield,22.250513,1225000.0,0.483333
3538,22/23,Attack,20.506502,400000.0,0.376667
